In [ ]:
# Display the Python interpreter used by the current Jupyter kernel.
# This is useful for confirming that the notebook is running inside the
# intended Conda environment before any analysis is started.
import sys
print(sys.executable)

In [ ]:
# Import the principal packages required by the bootstrap GMM workflow.
# This cell also reports key package versions so that the computational
# environment used for the analysis can be checked and documented.
import numpy as np
import pandas as pd
import sklearn
import joblib
import matplotlib
import openpyxl

from sklearn.mixture import GaussianMixture

print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)
print("所有依赖正常")

In [ ]:
import os
import sys
import subprocess
from pathlib import Path


# ============================================================
# 1. USER CONFIGURATION: MODIFY ONLY THIS SECTION FOR EACH RUN
# ============================================================
# All dataset-specific settings are defined here. The remaining sections
# validate these settings, pass them to the standalone Python script through
# environment variables, and execute the analysis. In routine use, only the
# paths, sample identifiers, output name, and computational parameters below
# need to be changed.

# Path to the standalone Python script that performs the complete bootstrap
# Gaussian mixture modeling analysis.
script_path = Path(
    "/media/tigerwp/data/Xiehongsen/Illustrate of Yangtze data/"
    "bootstrap GMM/Supplemental_bootstrap_GMM.py"
)

# Path to the input Excel workbook containing the geochronologic age data.
excel_path = Path(
    "/media/tigerwp/data/Xiehongsen/Illustrate of Yangtze data/"
    "baserock python data-1.xlsx"
)

# Directory in which all numerical results and figures will be written.
# The directory is created automatically below if it does not already exist.
output_dir = Path(
    "/media/tigerwp/data/Xiehongsen/Illustrate of Yangtze data/"
    "bootstrap GMM/new figure 60/"
)

# Sample_ID values for the two datasets to be analyzed. These strings must
# exactly match the corresponding entries in the Sample_ID column of the
# input Excel workbook.
sample_a = "Zr-Lower Yangtze"
sample_b = "Ar-Lower Yangtze"

# Prefix used for output files. Do not append file extensions such as .png,
# .csv, or .xlsx because the analysis script adds them automatically.
output_name = "Lower Yangtze-GMM"

# Main computational settings:
#   n_bootstrap : number of bootstrap replicates.
#   n_jobs      : number of parallel worker processes.
#   gmm_n_init  : number of independent initializations used for each GMM fit.
n_bootstrap = 5000
n_jobs = 64
gmm_n_init = 10


# ============================================================
# 2. FILE AND PARAMETER VALIDATION
# ============================================================
# Perform basic checks before launching the computationally intensive analysis.
# These checks are intended to detect missing files, invalid sample identifiers,
# or inappropriate numerical settings before the external script is started.

if not script_path.is_file():
    raise FileNotFoundError(f"找不到 Python 脚本：\n{script_path}")

if not excel_path.is_file():
    raise FileNotFoundError(f"找不到 Excel 文件：\n{excel_path}")

if not sample_a.strip():
    raise ValueError("sample_a 不能为空")

if not sample_b.strip():
    raise ValueError("sample_b 不能为空")

if sample_a.strip() == sample_b.strip():
    raise ValueError("sample_a 和 sample_b 不能相同")

if n_bootstrap < 1:
    raise ValueError("n_bootstrap 必须大于 0")

if n_jobs == 0:
    raise ValueError("n_jobs 不能为 0")

if gmm_n_init < 1:
    raise ValueError("gmm_n_init 必须大于 0")

# Create the output directory, including any missing parent directories.
output_dir.mkdir(parents=True, exist_ok=True)


# ============================================================
# 3. PASS THE CONFIGURATION TO THE PYTHON ANALYSIS SCRIPT
# ============================================================
# The standalone analysis script reads its run-specific settings from
# environment variables. This keeps the numerical implementation separate
# from the configuration used for a particular analysis.

# Start from a copy of the current environment so that the subprocess retains
# access to the active Conda/Python environment and other required variables.
env = os.environ.copy()

# Pass the input path, output directory, sample identifiers, and output prefix.
env["GMM_EXCEL_PATH"] = str(excel_path)
env["GMM_OUT_DIR"] = str(output_dir)
env["GMM_SAMPLE_A"] = sample_a.strip()
env["GMM_SAMPLE_B"] = sample_b.strip()
env["GMM_FNAME"] = output_name.strip()

# Pass the bootstrap, parallel-processing, and GMM initialization settings.
# Environment variables are strings, so numerical values are converted before
# they are passed to the standalone analysis script.
env["GMM_N_BOOTSTRAP"] = str(n_bootstrap)
env["GMM_N_JOBS"] = str(n_jobs)
env["GMM_N_INIT"] = str(gmm_n_init)

# Restrict low-level numerical libraries to one thread inside each parallel
# worker process. The outer bootstrap loop is parallelized with multiple Python
# processes; allowing BLAS/OpenMP libraries to spawn additional threads inside
# every worker could cause nested parallelism, CPU oversubscription, excessive
# memory use, and reduced computational efficiency.
env["OMP_NUM_THREADS"] = "1"
env["MKL_NUM_THREADS"] = "1"
env["OPENBLAS_NUM_THREADS"] = "1"
env["NUMEXPR_NUM_THREADS"] = "1"
env["VECLIB_MAXIMUM_THREADS"] = "1"
env["BLIS_NUM_THREADS"] = "1"

# Use a non-interactive Matplotlib backend. This prevents graphical windows
# from opening and blocking execution when the notebook is run on a remote
# server, headless Linux machine, or SSH session without a display.
env["MPLBACKEND"] = "Agg"


# ============================================================
# 4. DISPLAY THE SETTINGS USED FOR THIS RUN
# ============================================================
# Print the complete run configuration before execution. Keeping these values
# visible in the notebook output makes it easier to document, reproduce, and
# troubleshoot a specific analysis run.

print("本次运行设置")
print("-" * 60)
print("Python：", sys.executable)
print("脚本：", script_path)
print("Excel：", excel_path)
print("输出目录：", output_dir)
print("样品 A：", sample_a)
print("样品 B：", sample_b)
print("输出名称：", output_name)
print("Bootstrap：", n_bootstrap)
print("并行进程：", n_jobs)
print("GMM n_init：", gmm_n_init)
print("-" * 60)


# ============================================================
# 5. RUN THE COMPLETE ANALYSIS SCRIPT
# ============================================================
# Execute the standalone Python script as a subprocess. sys.executable ensures
# that the same Python interpreter used by this Jupyter kernel is also used to
# run the external script. check=False allows the return code to be inspected
# explicitly after execution rather than raising an immediate exception.

result = subprocess.run(
    [sys.executable, str(script_path)],
    env=env,
    check=False
)

print("-" * 60)
print("程序返回代码：", result.returncode)

# A conventional return code of 0 indicates successful completion. Any non-zero
# value indicates that the external script terminated with an error; in that
# case, the diagnostic messages printed above should be inspected.
if result.returncode == 0:
    print("计算正常完成。")
    print("结果保存在：", output_dir)
else:
    print("程序运行失败，请查看上方报错信息。")